# Who said what? A four-span development pilot

This pilot follows `docs/METHODS.md`. It prepares four selected claim spans from three previously inspected statements. These are development examples, not an independent test or a representative sample. No model is trained and no human label is suggested. Frozen v2 annotations remain unchanged.

The question concerns **expressed claims**, not actual guilt or innocence. Always read the complete statement: a grammatical first-person pronoun inside a reported claim does not by itself identify the source of that claim.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next((p for p in [current, *current.parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
source = ROOT / "data/processed/model_corpus_v1.csv"
manifest = json.loads((ROOT / "results/followup_audit_v1/manifest.json").read_text())
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
if source_hash != manifest["source_sha256"]:
    raise ValueError("Prepared corpus differs from the frozen follow-up input.")
corpus = pd.read_csv(source, keep_default_na=False)
if not corpus["execution_number"].is_unique:
    raise ValueError("Duplicate execution IDs.")
print("Verified frozen prepared-corpus hash. No models were fitted.")


## Review fields

Leave fields blank until reviewed; use `unclear` only after considering the context.

| Field | Values | Question |
|---|---|---|
| claim_source | self / other / unclear | Whose claim is being presented? |
| event_type | killing_action / knowledge / other / unclear | What event or state is affirmed or negated? |
| event_actor | self / other / unspecified / unclear | Who performs the action or holds the knowledge inside the claim? |
| polarity | affirmed / negated / unclear | Is that particular proposition affirmed or negated? |
| review_note | free text | What wording supports the reading, and what remains unresolved? |

These fields are separate. Source and actor need not be the same. A claim about not knowing is not automatically a denial of performing an act. The old eight labels are not gold labels for these new fields.


In [ ]:
# Purposively selected, previously inspected development spans.
# Selection does not provide reference annotations.
selections = [
    (117, "claim_1", "I did not kill anyone."),
    (391, "claim_1", "I didn't know anybody was there."),
    (499, "claim_1", "I didn't commit those murders"),
    (499, "claim_2", "I really did it."),
]
annotation_columns = ["claim_source", "event_type", "event_actor", "polarity", "review_note"]
rows = []
for number, local_id, anchor in selections:
    row = corpus.loc[corpus["execution_number"].eq(number)].iloc[0]
    context = row["model_text"]
    if context.count(anchor) != 1:
        raise ValueError(f"Expected exactly one span match for execution {number}.")
    start = context.index(anchor)
    end = start + len(anchor)
    record = {
        "claim_id": f"execution_{number}_{local_id}",
        "execution_number": number,
        "source_url": row["source_url"],
        "context_text": context,
        "span_start": start,
        "span_end": end,
        "claim_text": context[start:end],
        "context_sha256": hashlib.sha256(context.encode("utf-8")).hexdigest(),
        "corpus_sha256": source_hash,
        "selection_status": "previously_inspected_development_example",
    }
    record.update({column: "" for column in annotation_columns})
    rows.append(record)
pilot = pd.DataFrame(rows)
assert pilot["claim_id"].is_unique and len(pilot) == 4
print("Four spans from three documents; all annotation fields are blank.")
display(pilot[["claim_id", "execution_number", "claim_text"]])


In [ ]:
# Read the exact selected span and its complete prepared context.
for _, row in pilot.iterrows():
    print("\n" + "=" * 70)
    print("Claim:", row["claim_id"])
    print("Source:", row["source_url"])
    print("Selected span:", row["claim_text"])
    print("Offsets (zero-based, end-exclusive):", row["span_start"], row["span_end"])
    print("Full prepared statement:\n", row["context_text"])


## Save a separate pilot worksheet

The following cell creates a new ignored CSV under `data/processed/`. It does not modify the corpus or frozen annotations. If the worksheet already exists, source fields must match, and existing annotations are retained without overwriting them. This is a development worksheet, not a scored benchmark.


In [ ]:
target = ROOT / "data/processed/semantic_pilot_development_v1.csv"
source_columns = [column for column in pilot.columns if column not in annotation_columns]
if target.exists():
    existing = pd.read_csv(target, keep_default_na=False)
    if set(existing.columns) != set(pilot.columns) or not existing["claim_id"].is_unique:
        raise ValueError("Existing worksheet has an unexpected schema or duplicate IDs.")
    if set(existing["claim_id"]) != set(pilot["claim_id"]):
        raise ValueError("Existing worksheet has different claim IDs.")
    existing = existing.set_index("claim_id").loc[pilot["claim_id"]].reset_index()
    pd.testing.assert_frame_equal(
        existing[source_columns], pilot[source_columns], check_dtype=False,
    )
    print("Existing worksheet source fields match; annotations were not overwritten.")
else:
    pilot.to_csv(target, index=False, encoding="utf-8-sig")
    print("Created:", target.relative_to(ROOT))
print("No predictions, performance scores or new frozen labels were created.")
